# Optimizing LibreYOLO for Qualcomm® Snapdragon Devices Using QAIRT

This notebook walks through the full pipeline to optimize a **LibreYOLO** object detection model for efficient inference on Qualcomm® Snapdragon devices using the **Qualcomm AI Runtime (QAIRT) SDK**.

The pipeline covers:
1. **Image preprocessing** — resize and normalize images for model input.
2. **Calibration dataset preparation** — generate representative `.raw` samples for INT8 quantization.
3. **Model export** — load a pre-trained LibreYOLOXs checkpoint and export it to ONNX.
4. **DLC conversion** — convert the ONNX model to QAIRT's Deep Learning Container (DLC) format using `qairt-converter`.
5. **INT8 quantization** — apply post-training quantization for faster, more efficient inference using `qairt-quantizer`.
6. **HTP context binary compilation** — compile the DLC offline for the Hexagon Tensor Processor (HTP) on the Snapdragon 778G (sm7325) using `qairt-context-binary-generator`.

We begin by importing the necessary libraries.

In [ ]:
# Import necessary libraries.
import glob
import os
import random
import torch
import uuid

import numpy as np

from libreyolo import LibreYOLO
from PIL import Image

## Preprocessing and Calibration Data

Before converting or quantizing the model, two things are needed:

- A **preprocessing function** to transform raw images into the tensor format expected by LibreYOLO (resized to 640×640, normalized to `[0, 1]`).
- A **representative calibration dataset** in `.raw` binary format. QAIRT's `qairt-quantizer` uses this dataset during post-training quantization to compute per-layer scale factors for INT8 weights and activations.

The cell below defines the `preprocess()` helper function used throughout this pipeline.

In [ ]:
def preprocess(original_image: np.ndarray, size: int = 640) -> np.ndarray:
    """
    Preprocess the input image for model inference.

    Args:
        original_image (np.ndarray): Input image as NumPy array.
        size (int): Target image size.

    Returns:
        np.ndarray: Preprocessed float32 image normalized to [0, 1].
    """

    image = Image.fromarray(original_image).convert("RGB")
    image = image.resize((size, size), Image.BILINEAR)

    image = np.asarray(image).astype(np.float32) / 255.0
    image = np.transpose(image, (2, 0, 1))  # HWC -> CHW
    image = np.expand_dims(image, axis=0)   # CHW -> NCHW

    return image

### Preparing the Calibration Dataset

QAIRT's quantization tool (`qairt-quantizer`) requires input samples in `.raw` format — flat binary files containing `float32` pixel values with shape `(H, W, C)`.

The code below:
1. Downloads the **COCO 2017 validation set** (~777 MB).
2. Randomly samples **100 images** (with a fixed seed for reproducibility).
3. Preprocesses each image using `preprocess()` and saves it as a `.raw` file.
4. Generates a `filenames.txt` manifest listing all `.raw` files — required by `qairt-quantizer`.

In [ ]:
if not os.path.exists("val"):
    !bash -c 'curl -L -o val2017.zip http://images.cocodataset.org/zips/val2017.zip'
    !bash -c 'unzip val2017.zip -d val'
    !bash -c 'rm val2017.zip'

if not os.path.exists("raw"):
    !bash -c 'mkdir raw'

    random.seed(42)
    SAMPLE_SIZE = 100

    filenames = glob.glob(f"val/**/*.jpg", recursive=True)
    random.shuffle(filenames)

    for filename in filenames[:SAMPLE_SIZE]:
        image = Image.open(filename)
        original_image = np.array(image)
        processed_image = preprocess(original_image)
        processed_image.tofile(f"raw/{uuid.uuid4()}.raw")

if not os.path.exists("raw/filenames.txt"):
    !bash -c 'find raw -name "*.raw" > raw/filenames.txt'

### Loading LibreYOLO and Exporting to ONNX

QAIRT does not consume LibreYOLO models in PyTorch format directly. The model must first be exported to the **ONNX (Open Neural Network Exchange)** format, which `qairt-converter` can then parse and convert to its DLC format.

The code below:
1. Downloads the pre-trained **LibreYOLOXs** checkpoint from Hugging Face.
2. Loads it using the `LibreYOLO` wrapper and sets it to evaluation mode.
3. Exports it to ONNX using `torch.onnx.export()` with `opset_version=18`.

The model produces **three output heads** corresponding to different detection scales:
- `output_small` (80×80) — detects small objects.
- `output_medium` (40×40) — detects medium objects.
- `output_large` (20×20) — detects large objects.

In [ ]:
if not os.path.exists("../models/LibreYOLOXs.pt"):
    !bash -c 'curl -L -o ../models/LibreYOLOXs.pt https://huggingface.co/LibreYOLO/LibreYOLOXs/resolve/main/LibreYOLOXs.pt?download=true'

yolo = LibreYOLO("../models/LibreYOLOXs.pt")
torch_model = yolo.model
torch_model.eval()

if not os.path.exists("../models/LibreYOLOXs.onnx"):
    dummy = torch.randn(1, 3, 640, 640)
    torch.onnx.export(
        torch_model,
        dummy,
        "../models/LibreYOLOXs.onnx",
        opset_version=18,
        dynamo=False,
        input_names=["images"],
        output_names=[
            "output_small",   # 80x80
            "output_medium",  # 40x40
            "output_large"    # 20x20
        ]
    )

## Converting the Model to QAIRT DLC Format

QAIRT uses the **DLC (Deep Learning Container)** format as an intermediate representation. The first step is to convert the ONNX model to a floating-point (**FP32**) DLC using the `qairt-converter` tool. This DLC file is the starting point for all subsequent quantization and compilation steps.

In [ ]:
!bash -c 'qairt-converter \
    --input_network ../models/LibreYOLOXs.onnx \
    --output_path ../models/qairt/LibreYOLOXs_fp32.dlc'

### Inspecting the FP32 DLC

The `qairt-dlc-info` command prints a detailed summary of the DLC graph, including layer names, types, input/output tensor shapes, and supported execution backends (CPU, GPU, HTP). Reviewing this output verifies that the ONNX export was clean and that all operators are supported by QAIRT before proceeding to quantization.

In [ ]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc'

### INT8 Post-Training Quantization

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. The `qairt-quantizer` tool applies **post-training quantization (PTQ)** by computing per-layer scale factors from the calibration `.raw` samples prepared earlier, producing an INT8 DLC.

In [ ]:
!bash -c 'qairt-quantizer \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc \
    --input_list raw/filenames.txt \
    --output_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

### Inspecting the INT8 DLC

After quantization, `qairt-dlc-info` is used again to inspect the INT8 DLC. Compared to the FP32 output, you should observe that layer types now reflect quantized variants. The INT8 DLC is expected to be smaller than the FP32 DLC, but the actual reduction should be measured.

In [ ]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

### Compiling the Graph for the Hexagon Tensor Processor (HTP)

Qualcomm® Snapdragon SoCs include a dedicated AI accelerator called the **Hexagon Tensor Processor (HTP)**. To fully leverage HTP hardware, QAIRT can perform an **offline context binary compilation** step that optimizes the DLC graph layout for a specific target SoC.

The `qairt-context-binary-generator` command takes the INT8 DLC and the target SoC identifier — `sm7325`, corresponding to the **Snapdragon 778G** — and produces a pre-compiled context binary (`.bin`) in the specified output directory. This binary is ready for on-device deployment with maximum HTP utilization.

In [ ]:
!bash -c 'qairt-context-binary-generator \
    --dlc_path ../models/qairt/LibreYOLOXs_int8.dlc \
    --output_dir ../models/qairt/sm7325/ \
    --htp_socs sm7325'

### Verifying the HTP Context Binary

Unlike DLC files, the compiled context binary (`.bin`) produced by `qairt-context-binary-generator` cannot be inspected with `qairt-dlc-info`. Instead, a directory listing of the output folder confirms that the binary was generated successfully and shows its size on disk.

In [ ]:
!bash -c 'ls -lh ../models/qairt/sm7325/'

By following these steps, the model is optimized for efficient inference on Qualcomm® platforms, leveraging hardware acceleration for real-time AI applications. This process ensures that the model is both accurate and performant, making it suitable for deployment in edge devices powered by Qualcomm® chipsets.